In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

train_tx = pd.read_csv('../data/raw/train_transaction.csv')
train_id = pd.read_csv('../data/raw/train_identity.csv')

df = train_tx.merge(train_id, on='TransactionID', how='left')

cols_to_drop = ['id_24', 'id_25', 'id_07', 'id_08', 'id_21', 'id_26', 'id_22', 'id_27', 'id_23', 'dist2', 'D7', 'id_18']
df = df.drop(columns=cols_to_drop)

print(df.shape)

(590540, 422)


In [2]:
df = df.sort_values('TransactionDT').reset_index(drop=True)

split_point = int(len(df) * 0.8)
train_df = df.iloc[:split_point].copy()
val_df = df.iloc[split_point:].copy()

print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"\nTrain date range: {train_df['TransactionDT'].min()} to {train_df['TransactionDT'].max()}")
print(f"Val date range: {val_df['TransactionDT'].min()} to {val_df['TransactionDT'].max()}")
print(f"\nTrain fraud rate: {train_df['isFraud'].mean():.4f}")
print(f"Val fraud rate: {val_df['isFraud'].mean():.4f}")

Train shape: (472432, 422)
Val shape: (118108, 422)

Train date range: 86400 to 12192842
Val date range: 12192900 to 15811131

Train fraud rate: 0.0351
Val fraud rate: 0.0344


In [3]:
v_cols = [c for c in df.columns if c.startswith('V')]

v_medians = train_df[v_cols].median()

train_df[v_cols] = train_df[v_cols].fillna(v_medians)
val_df[v_cols] = val_df[v_cols].fillna(v_medians)

print("V columns still missing in train:", train_df[v_cols].isnull().sum().sum())
print("V columns still missing in val:", val_df[v_cols].isnull().sum().sum())

V columns still missing in train: 0
V columns still missing in val: 0


In [4]:
exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT']

remaining_cols = [c for c in df.columns if c not in v_cols and c not in exclude_cols]

numeric_cols = train_df[remaining_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = train_df[remaining_cols].select_dtypes(include=['object']).columns.tolist()

print(f"Remaining numeric columns: {len(numeric_cols)}")
print(numeric_cols)
print(f"\nRemaining categorical columns: {len(categorical_cols)}")
print(categorical_cols)

Remaining numeric columns: 51
['TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_09', 'id_10', 'id_11', 'id_13', 'id_14', 'id_17', 'id_19', 'id_20', 'id_32']

Remaining categorical columns: 29
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


/tmp/ipykernel_5805/1715667180.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train_df[remaining_cols].select_dtypes(include=['object']).columns.tolist()


In [5]:
numeric_medians = train_df[numeric_cols].median()

train_df[numeric_cols] = train_df[numeric_cols].fillna(numeric_medians)
val_df[numeric_cols] = val_df[numeric_cols].fillna(numeric_medians)

print("Numeric columns still missing in train:", train_df[numeric_cols].isnull().sum().sum())
print("Numeric columns still missing in val:", val_df[numeric_cols].isnull().sum().sum())

Numeric columns still missing in train: 0
Numeric columns still missing in val: 0


In [6]:
train_df[categorical_cols] = train_df[categorical_cols].fillna('missing')
val_df[categorical_cols] = val_df[categorical_cols].fillna('missing')

print("Categorical columns still missing in train:", train_df[categorical_cols].isnull().sum().sum())
print("Categorical columns still missing in val:", val_df[categorical_cols].isnull().sum().sum())

Categorical columns still missing in train: 0
Categorical columns still missing in val: 0


In [7]:
low_card_cols = ['ProductCD', 'card4', 'card6']

train_df = pd.get_dummies(train_df, columns=low_card_cols, prefix=low_card_cols)
val_df = pd.get_dummies(val_df, columns=low_card_cols, prefix=low_card_cols)

# Align columns in case a category exists in one set but not the other
train_df, val_df = train_df.align(val_df, join='left', axis=1, fill_value=0)

print(train_df.shape, val_df.shape)

(472432, 434) (118108, 434)


In [8]:
remaining_categorical = [c for c in categorical_cols if c not in low_card_cols]

for col in remaining_categorical:
    freq_map = train_df[col].value_counts(normalize=True)
    train_df[col + '_freq'] = train_df[col].map(freq_map)
    val_df[col + '_freq'] = val_df[col].map(freq_map).fillna(0)  # unseen categories in val get 0

train_df = train_df.drop(columns=remaining_categorical)
val_df = val_df.drop(columns=remaining_categorical)

print(train_df.shape, val_df.shape)
print("\nAny remaining object/string columns?")
print(train_df.select_dtypes(include=['object']).columns.tolist())

/tmp/ipykernel_5805/2160958778.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[col + '_freq'] = train_df[col].map(freq_map)
/tmp/ipykernel_5805/2160958778.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_df[col + '_freq'] = val_df[col].map(freq_map).fillna(0)  # unseen categories in val get 0


(472432, 434) (118108, 434)

Any remaining object/string columns?
[]


In [9]:
print("Total NaNs in train_df:", train_df.isnull().sum().sum())
print("Total NaNs in val_df:", val_df.isnull().sum().sum())

print("\nDtype summary:")
print(train_df.dtypes.value_counts())

Total NaNs in train_df: 0
Total NaNs in val_df: 0

Dtype summary:
float64    415
bool        15
int64        4
Name: count, dtype: int64


In [10]:
import os

os.makedirs('../data/processed', exist_ok=True)

train_df.to_csv('../data/processed/train_processed.csv', index=False)
val_df.to_csv('../data/processed/val_processed.csv', index=False)

print("Saved.")
print(f"train_processed.csv: {os.path.getsize('../data/processed/train_processed.csv') / 1e6:.1f} MB")
print(f"val_processed.csv: {os.path.getsize('../data/processed/val_processed.csv') / 1e6:.1f} MB")

Saved.
train_processed.csv: 1072.9 MB
val_processed.csv: 267.9 MB


In [14]:
import joblib

joblib.dump(train_df, '../data/processed/train_processed.pkl', compress=3)
joblib.dump(val_df, '../data/processed/val_processed.pkl', compress=3)

os.remove('../data/processed/train_processed.csv')
os.remove('../data/processed/val_processed.csv')

print(f"train_processed.pkl: {os.path.getsize('../data/processed/train_processed.pkl') / 1e6:.1f} MB")
print(f"val_processed.pkl: {os.path.getsize('../data/processed/val_processed.pkl') / 1e6:.1f} MB")

train_processed.pkl: 60.2 MB
val_processed.pkl: 15.7 MB
